# Ⅰ第1回 演習3（E1-3）「良いAI」を定義して順位を出す

演習2 の 4 指標に **班で決めた重み** を付け，3 モデルの順位を出す．
教科書 5.3 節「クラス別性能評価」が正解率を分解して見たのと同じ発想で，ここでは正解率以外の 3 指標も同時に見る．
重みの付け方は班の自由．ただし **なぜその重みか** を一文で書く．

同じ数値でも班が違えば順位が変わる．それを次のプレゼンで他班の結果と比較する．
実行時間の目安は 5 分（計算はすぐ終わる．時間は議論に使う）．


## (0) 班と役割の設定

In [ ]:
# ===== (0) 班と役割の設定 =====
GROUP_ID = 1                  # ← 自分の班番号 (1〜27) に書き換える
MEMBER_ROLE = "implementer"   # implementer / verifier / recorder / presenter（5 人班は collector も）のいずれか．3 人班で presenter を兼ねる verifier は "verifier"

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) コードテンプレート①：ライブラリのインポート

In [ ]:
import itertools
import numpy as np
import pandas as pd
from common import RESULTS
from common.logger import ResultLogger, read_results
from common.device import get_device
device = get_device()

## (2) 演習2 の結果を読む

共通形式の CSV から自班の行だけを取り，モデル × 指標の表に戻す．

In [ ]:
df = read_results(course="c1", day="d1", exercise="ex2")
df = df[df["group_id"] == f"{GROUP_ID:02d}"]
assert len(df) > 0, "演習2 の結果がない．先に ex2.ipynb を最後まで実行すること"
# 同じ班が複数回実行していれば最新の値を使う
df = df.sort_values("timestamp").groupby(["condition", "metric_name"]).tail(1)
table = df.pivot(index="condition", columns="metric_name", values="metric_value")
table.index.name = "model"
METRICS = ["accuracy", "loss", "infer_ms_per_1000", "peak_mem_mb"]     # 必須 4 指標
# 任意: "latency_ms_b1" を METRICS に足してもよい（5 指標になる）
table[METRICS].round(4)

## (3) 指標の向き

「大きいほど良い」指標と「小さいほど良い」指標が混ざっている．どちらかを間違えると順位が反転する．

In [ ]:
# TODO: 各指標が「高いほど良い(higher)」か「低いほど良い(lower)」かを書く
DIRECTION = {
    "accuracy":          ...,
    "loss":              ...,
    "infer_ms_per_1000": ...,
    "peak_mem_mb":       ...,
    "latency_ms_b1":     ...,
}
assert all(m in DIRECTION for m in METRICS)
assert all(v in ("higher", "lower") for v in DIRECTION.values()), "TODO 未実装: 値は 'higher' か 'lower' の文字列にする"

## (4) 正規化：単位の違う指標を足せる形にする

% と ms と MB は足せない．揃え方は 1 つではない．

- **minmax**：3 モデル中の最悪 = 0，最良 = 1 に線形に引き伸ばす．差の大きさが反映される
- **rank**：順位だけを使う（1 位 = 1，2 位 = 0.5，3 位 = 0）．差の大きさは捨てる

どちらも「正しい」．しかし **同じ重みでも順位が変わりうる**．

In [ ]:
def normalize(table, metrics, direction, method="minmax"):
    out = pd.DataFrame(index=table.index)
    for m in metrics:
        x = table[m].astype(float)
        if method == "minmax":
            rng = x.max() - x.min()
            z = (x - x.min()) / rng if rng > 0 else pd.Series(0.5, index=x.index)
            # TODO: direction[m] が "lower" なら z を反転して「最良 = 1」にする
            ...
        elif method == "rank":
            r = x.rank(ascending=(direction[m] == "lower"), method="min")   # 最良 = 1 位
            z = 1.0 - (r - 1) / (len(x) - 1)     # 1 位 = 1, 最下位 = 0
        else:
            raise ValueError(method)
        out[m] = z
    return out

print("minmax"); display(normalize(table, METRICS, DIRECTION, "minmax").round(3))
print("rank");   display(normalize(table, METRICS, DIRECTION, "rank").round(3))

## (5) グループディスカッション（10 分）：班の「良いAI」の定義と重み

**進め方**：(a) 4 人が各自 1 分で「このモデルを誰がどこで使うか」を 1 つ挙げる → (b) 班で 1 つの用途に絞る → (c) その用途に合う定義を **一文** で書き，4 指標の重み（合計 1）と正規化（minmax / rank）を決める．
**必ず理由を言葉にする**：「なぜ正解率より速度が重いのか」を他班に説明できる形にしておく．

例：「学生のスマホで動く映画レビュー判定器」なら速度とメモリが重い．「研究用の最高正解率」なら正解率だけ．

In [ ]:
# TODO: 班の定義を一文で書き，正規化の方法を選び，4 指標に重みを付ける（合計 1）
DEFINITION = "..."
METHOD = ...                 # "minmax" か "rank"
WEIGHTS = {
    "accuracy":          ...,
    "loss":              ...,
    "infer_ms_per_1000": ...,
    "peak_mem_mb":       ...,
}
assert DEFINITION != "..." and METHOD is not ..., "TODO 未実装: 定義と正規化の方法を書く"
assert all(isinstance(w, (int, float)) for w in WEIGHTS.values()), "TODO 未実装: 4 指標に数値の重みを付ける"
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "重みの合計を 1 にする"
assert set(WEIGHTS) == set(METRICS), "METRICS と同じ指標に重みを付ける"
assert METHOD in ("minmax", "rank")
print(DEFINITION); print(METHOD, WEIGHTS)

## (6) スコアと順位

In [ ]:
def rank_models(table, weights, method):
    norm = normalize(table, list(weights), DIRECTION, method)
    w = pd.Series(weights)
    score = (norm[w.index] * w).sum(axis=1)
    return pd.DataFrame({"score": score, "rank": score.rank(ascending=False, method="min").astype(int)}).sort_values("rank")

result = rank_models(table, WEIGHTS, METHOD)
result.round(3)

## (7) 感度分析：重みと正規化をどれだけ動かすと 1 位が変わるか

他班と順位が違うのは，重みか正規化の方法が違うから．どこで順位が入れ替わるかを先に見ておくと，プレゼンでの反論に備えられる．

In [ ]:
profiles = {
    "正解率最優先": {"accuracy": 0.7, "loss": 0.1, "infer_ms_per_1000": 0.1, "peak_mem_mb": 0.1},
    "速度最優先": {"accuracy": 0.1, "loss": 0.1, "infer_ms_per_1000": 0.7, "peak_mem_mb": 0.1},
    "省メモリ":   {"accuracy": 0.1, "loss": 0.1, "infer_ms_per_1000": 0.1, "peak_mem_mb": 0.7},
    "均等":       {"accuracy": 0.25, "loss": 0.25, "infer_ms_per_1000": 0.25, "peak_mem_mb": 0.25},
    "自班":       WEIGHTS,
}
sens = pd.DataFrame({f"{name}/{meth}": rank_models(table, w, meth)["rank"]
                     for name, w in profiles.items() for meth in ("minmax", "rank")})
print("順位表（行 = モデル，列 = 重みの付け方 / 正規化）")
display(sens)
print("異なる順位の数:", sens.T.apply(tuple, axis=1).nunique(), "/", sens.shape[1])

# 正解率の重みを 0→1 に動かし，残りを均等に配ったとき 1 位がどこで変わるか（正規化 2 通り）
flips = {}
for meth in ("minmax", "rank"):
    first = []
    for wa in np.linspace(0, 1, 21):
        rest = (1 - wa) / 3
        w = {"accuracy": wa, "loss": rest, "infer_ms_per_1000": rest, "peak_mem_mb": rest}
        first.append((round(wa, 2), rank_models(table, w, meth).index[0]))
    flips[meth] = first
    print(f"[{meth}] 正解率の重み → 1位:", " ".join(f"{wa}:{m}" for wa, m in first))

## (8) CSV に記録

順位・スコア・重みを共通形式で残す．`aggregate/c1d1_rank.py` が提出された班の分を1枚にする．

In [ ]:
logger = ResultLogger(GROUP_ID, MEMBER_ROLE, course="c1", day="d1", exercise="ex3", device=device)
for arch, r in result.iterrows():
    logger.log_many({"rank": r["rank"], "score": r["score"]}, condition=arch)
for m, w in WEIGHTS.items():
    logger.log(f"weight_{m}", w, condition="weights")
logger.log("method_is_rank", 1.0 if METHOD == "rank" else 0.0, condition=METHOD)
(RESULTS / f"c1_d1_ex3_definition_{logger.group_id}.txt").write_text(DEFINITION + "\n", encoding="utf-8")
print("書き込み先:", logger.path)

## (9) グループ内プレゼン3（1 人 2 分）用まとめ

スライドは作らない．この出力を画面に出して話す．

**話す内容**：implementer = 定義と重みの根拠 ／ verifier = 重みか正規化を変えたとき順位がどう動いたか（(7) の表）／ recorder = 討議で割れた点と決め方 ／ auditor = 他班に投げる質問 3 問（`waiting_task.md` の評価基準）

In [ ]:
print("班", logger.group_id, "|", DEFINITION)
print("正規化:", METHOD, "| 重み:", WEIGHTS)
print(result.round(3).to_string())
for meth, first in flips.items():
    chg = [f"正解率の重み {wa} 以上で {m}" for wa, m in first if m != first[0][1]][:1]
    print(f"[{meth}] 1 位が変わる条件:", chg[0] if chg else "この範囲では変わらない")

## (10) 班間ディスカッション：質問カード交換（20 分）と全体共有

1. auditor の質問 3 問を別班へ渡す（同じ教室内）．受け取った班は **自班の実測値を使って** 10 分で答える
2. 質問元が 3 段階で採点する（3 = 実測値を要求する質問，2 = 手法の理解を問う，1 = 定義を問うだけ）
3. 回答を受け取る際に，相手班の **演習2 の表（`table_c1_d1_ex2_<班>.csv`）と演習3 の順位・重み** を受け取る（口頭・AirDrop どちらでも）．**3〜4 班分** 集めてレポートで自班と比べる．提出された班の統合 CSV は授業後に教員が配る
4. 教員が `aggregate/c1d1_rank.py` の図を投影する．**同じ 3 モデルの数値から何通りの順位が出たか** を確認し，順位が割れた理由（重みか，正規化か）を 2 班に発言してもらう

**次回に持ち越す問い**：ほとんどの班で A1 が 1 位になったとすれば，それは A1 が「良い」からか，それとも正解率の差 0.4 ポイントがばらつきの範囲内に収まっているからか．→ 第4回「再現性と分散」で確かめる．

In [ ]:
# 他班に渡す質問カード（auditor が記入して results/ に残す）
questions = [
    "Q1（段階3の例）: あなたの班の重みで正解率の重みを 0.1 下げたとき 1 位は変わりますか．変わるなら何から何へ",
    "Q2: ________",
    "Q3: ________",
]
card = RESULTS / f"c1_d1_ex3_questions_{logger.group_id}.txt"
card.write_text("\n".join(questions) + "\n", encoding="utf-8")
print("質問カード:", card)
print("\n".join(questions))